<a href="https://colab.research.google.com/github/kamalrajarulprakasam-sudo/GENAI-GOLD-Badge-Assignments/blob/main/Problem2_CUDA_Cpp_LLM_Inference/Problem2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 2 — C++ LLM Inference on NVIDIA GPUs with CUDA (Colab)

Self-contained Colab version of `Problem2_CUDA_Cpp_LLM_Inference`. It builds a minimal C++ program on top of **llama.cpp** that loads a quantized GGUF model and runs text generation offloaded to an NVIDIA GPU via **CUDA**.

> **Required:** `Runtime > Change runtime type > Hardware accelerator > GPU (T4)` before running this notebook. The build compiles CUDA kernels and can take 5–10 minutes the first time.

## 1. Confirm a GPU + CUDA toolkit are available

In [1]:
!nvidia-smi


Thu Sep 17 10:51:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## 2. Install build tools

In [2]:
!apt-get -qq update && apt-get -qq install -y cmake ninja-build


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...


## 3. Write out the project source files

In [3]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.18)
project(cuda_llm_inference LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

# Toggle NVIDIA GPU acceleration. Requires the CUDA Toolkit (nvcc) to be
# installed and discoverable by CMake. When OFF (or when no CUDA toolkit is
# found), llama.cpp falls back to a plain CPU build so this project still
# compiles and runs -- just without GPU offload.
option(GGML_CUDA "Build llama.cpp with NVIDIA CUDA GPU acceleration" ON)

include(FetchContent)
FetchContent_Declare(
    llama_cpp
    GIT_REPOSITORY https://github.com/ggml-org/llama.cpp.git
    GIT_TAG        b4079          # known-good tag; bump if you need newer model support
    GIT_SHALLOW    TRUE
)

# Skip llama.cpp's own examples/tests/server -- we only need the `llama`
# (+ ggml) library target for this minimal single-file demo.
set(LLAMA_BUILD_EXAMPLES OFF CACHE BOOL "" FORCE)
set(LLAMA_BUILD_TESTS    OFF CACHE BOOL "" FORCE)
set(LLAMA_BUILD_SERVER   OFF CACHE BOOL "" FORCE)
set(BUILD_SHARED_LIBS    OFF CACHE BOOL "" FORCE)

FetchContent_MakeAvailable(llama_cpp)

add_executable(cuda_llm_infer main.cpp)
target_link_libraries(cuda_llm_infer PRIVATE llama)

if(GGML_CUDA)
    message(STATUS "cuda_llm_infer: building with NVIDIA CUDA GPU offload enabled (GGML_CUDA=ON)")
else()
    message(STATUS "cuda_llm_infer: building CPU-only (GGML_CUDA=OFF)")
endif()


Writing CMakeLists.txt


In [4]:
%%writefile main.cpp
// cuda_llm_infer.cpp
//
// A minimal, single-file C++ program that loads a local GGUF LLM (e.g. a
// quantized Llama 3.2 model) and runs text-generation inference on an
// NVIDIA GPU via CUDA, using llama.cpp's `ggml-cuda` backend.
//
// Build & run instructions: see README.md in this folder.
//
// Usage:
//   cuda_llm_infer <model.gguf> "<prompt>" [n_predict=128] [n_gpu_layers=999]
//
//   n_gpu_layers = 999 offloads every transformer layer to the GPU (fastest,
//   requires enough VRAM). Use a smaller number (e.g. 20) to offload only
//   part of the model if it doesn't fit in VRAM, or 0 to force CPU-only.

#include "llama.h"

#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>
#include <vector>

namespace {

[[noreturn]] void fail(const std::string &msg) {
    std::fprintf(stderr, "Error: %s\n", msg.c_str());
    std::exit(1);
}

} // namespace

int main(int argc, char **argv) {
    if (argc < 3) {
        std::fprintf(
            stderr,
            "Usage: %s <model.gguf> \"<prompt>\" [n_predict=128] [n_gpu_layers=999]\n",
            argv[0]);
        return 1;
    }

    const std::string model_path = argv[1];
    const std::string prompt = argv[2];
    const int n_predict = argc > 3 ? std::atoi(argv[3]) : 128;
    const int n_gpu_layers = argc > 4 ? std::atoi(argv[4]) : 999;

    // 1. Initialize the ggml/llama backends. When llama.cpp was built with
    //    GGML_CUDA=ON and an NVIDIA GPU + driver is present, the CUDA
    //    backend is registered automatically and used for any layers we
    //    offload below.
    llama_backend_init();

    // 2. Load the GGUF model, offloading `n_gpu_layers` transformer layers
    //    to the GPU.
    llama_model_params model_params = llama_model_default_params();
    model_params.n_gpu_layers = n_gpu_layers;

    llama_model *model = llama_model_load_from_file(model_path.c_str(), model_params);
    if (!model) {
        fail("failed to load model from '" + model_path + "'");
    }

    const llama_vocab *vocab = llama_model_get_vocab(model);

    // 3. Create an inference context (KV cache size, batch size, ...).
    llama_context_params ctx_params = llama_context_default_params();
    ctx_params.n_ctx = 2048;
    ctx_params.n_batch = 512;

    llama_context *ctx = llama_init_from_model(model, ctx_params);
    if (!ctx) {
        fail("failed to create llama context");
    }

    // 4. Tokenize the prompt. llama_tokenize returns a negative "required
    //    size" if the buffer we pass is too small, so we retry once.
    std::vector<llama_token> tokens(prompt.size() + 8);
    int n_tokens = llama_tokenize(vocab, prompt.c_str(), (int32_t)prompt.size(),
                                   tokens.data(), (int32_t)tokens.size(),
                                   /*add_special=*/true, /*parse_special=*/true);
    if (n_tokens < 0) {
        tokens.resize(-n_tokens);
        n_tokens = llama_tokenize(vocab, prompt.c_str(), (int32_t)prompt.size(),
                                   tokens.data(), (int32_t)tokens.size(), true, true);
    }
    tokens.resize(n_tokens);

    // 5. Prefill: feed every prompt token through the model in one batch.
    llama_batch batch = llama_batch_get_one(tokens.data(), (int32_t)tokens.size());
    if (llama_decode(ctx, batch) != 0) {
        fail("llama_decode failed while processing the prompt");
    }

    // 6. Greedy-decode one token at a time until n_predict tokens are
    //    generated or the model produces an end-of-generation token.
    llama_sampler *sampler = llama_sampler_chain_init(llama_sampler_chain_default_params());
    llama_sampler_chain_add(sampler, llama_sampler_init_greedy());

    std::printf("Prompt: %s\n\n", prompt.c_str());
    std::printf("Running on: %s\n", n_gpu_layers > 0 ? "NVIDIA GPU (CUDA)" : "CPU");
    std::printf("Response:\n");

    for (int i = 0; i < n_predict; ++i) {
        llama_token new_token = llama_sampler_sample(sampler, ctx, -1);

        if (llama_vocab_is_eog(vocab, new_token)) {
            break;
        }

        char piece[256];
        int n = llama_token_to_piece(vocab, new_token, piece, sizeof(piece), 0, true);
        if (n > 0) {
            std::fwrite(piece, 1, n, stdout);
            std::fflush(stdout);
        }

        llama_batch next_batch = llama_batch_get_one(&new_token, 1);
        if (llama_decode(ctx, next_batch) != 0) {
            fail("llama_decode failed during generation");
        }
    }
    std::printf("\n");

    llama_sampler_free(sampler);
    llama_free(ctx);
    llama_model_free(model);
    llama_backend_free();
    return 0;
}


Writing main.cpp


## 4. Configure + build (GGML_CUDA=ON)

This fetches llama.cpp via CMake `FetchContent` and compiles it with CUDA support.

In [5]:
!mkdir -p build && cd build && \
  cmake .. -G Ninja -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON && \
  cmake --build . --config Release -j


-- The CXX compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- The C compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Found Git: /usr/bin/git (found version "2.43.0")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- OpenMP found
-- Using llamafile
-- Using AMX
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- CUDA found
-- Using CUDA architect

## 5. Download a small GGUF model (Llama 3.2 1B, ~0.8GB quantized)

In [6]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id='bartowski/Llama-3.2-1B-Instruct-GGUF',
    filename='Llama-3.2-1B-Instruct-Q4_K_M.gguf',
    local_dir='models',
)
print('Downloaded to', model_path)


Llama-3.2-1B-Instruct-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B /  808MB            

Llama-3.2-1B-Instruct-Q4_K_M.gguf: downloading bytes:           |  0.00B            

Downloaded to /content/models/Llama-3.2-1B-Instruct-Q4_K_M.gguf


## 6. Run inference on the GPU

In [7]:
!./build/cuda_llm_infer models/Llama-3.2-1B-Instruct-Q4_K_M.gguf "Explain what CUDA is in two sentences." 128 999


/bin/bash: line 1: ./build/cuda_llm_infer: No such file or directory


### Notes
- `n_gpu_layers=999` (the 4th CLI arg) offloads every transformer layer to the GPU. Pass `0` instead to force CPU-only.
- If `nvcc`/`nvidia-smi` are unavailable, re-run with a GPU runtime, or rebuild with `-DGGML_CUDA=OFF` for a CPU-only build.
- See the sibling `README.md` for troubleshooting (offline FetchContent, VRAM limits, etc.).